In [ ]:
%env CUDA_DEVICE_ORDER=
%env CUDA_VISIBLE_DEVICES=

In [ ]:
!uv pip install -q "kokoro>=0.9.2" soundfile torch
# !apt-get -qq -y install espeak-ng > /dev/null 2>&1

In [ ]:
import soundfile as sf
import torch
from IPython.display import Audio, display
from kokoro import KPipeline

In [ ]:
text = """
This is the CCT Estimator — a near-real-time stability monitor for grid operators — developed within tef T-S-O group.

CCT, Critical Clearing Time, is the longest a fault can stay on the grid and still be cleared while the system stays stable. The shorter it is, the thinner the stability margin.

From the grid's current state, the tool uses AI to estimate CCT for every monitored fault location — buses and lines alike — from past records and simulations. Each generator's predicted value is the orange diamond, with the full breakdown in the table below.

Anything sinking toward the red zone, under the stability threshold, is a spot where even a brief fault could set off a cascade.

So at a glance: which generators are most exposed, and which fault locations are dangerous right now.
""".strip()

In [ ]:
import numpy as np

SAMPLE_RATE = 24000
SILENCE = np.zeros(int(SAMPLE_RATE * 0.2), dtype=np.float32)  # 200ms gap between chunks

pipeline = KPipeline(lang_code="a", repo_id="hexgrad/Kokoro-82M")

In [ ]:
generator = pipeline(text, voice="am_michael", speed=0.95)

chunks = []
for i, (gs, _ps, audio) in enumerate(generator):
    print(i, gs)
    chunks.append(audio)
    sf.write(f"voiceover-pt{i}.wav", audio, SAMPLE_RATE)
    chunks.append(SILENCE)

combined = np.concatenate(chunks[:-1])  # drop trailing silence
display(Audio(data=combined, rate=SAMPLE_RATE, autoplay=True))

In [ ]:
sf.write("voiceover.wav", combined, SAMPLE_RATE)
print(f"Saved voiceover.wav  ({len(combined) / SAMPLE_RATE:.1f}s)")

In [ ]:
text = """
In this short presentation, we share our progress from the federated Testing and Experimentation Facility node focused on AI services for transmission system operators. Shortly, tef-T-S-O.

tef-T-S-O tackles everyday operator problems: fault and outage diagnostics, asset and infrastructure monitoring, grid planning and stability assessment, and coordinated control of flexibility resources. The aim is to turn operational data into tools that help operators respond to outages faster and plan more reliably.

Here we present the first completed service, Service 3: Dynamic AI-Enhanced Transmission Grid Stability Assessment. It gives operators a picture of the grid's transient stability, based on historical data and prior simulations.

This matters because conventional fault simulations can take up to 40 minutes to run. Instead of simulating, the service uses AI to estimate transient stability from similar past states — fast enough to support decisions in near real time.

Thanks for watching! Like and subscribe to EnerTEF on social media.

""".strip()

pipeline = KPipeline(lang_code="a", repo_id="hexgrad/Kokoro-82M")
generator = pipeline(text, voice="am_michael", speed=1.2)

chunks = []
for i, (gs, _ps, audio) in enumerate(generator):
    print(i, gs)
    chunks.append(audio)
    chunks.append(SILENCE)
    sf.write(f"voiceover-intro-pt{i}.wav", audio, SAMPLE_RATE)

combined = np.concatenate(chunks[:-1])  # drop trailing silence
display(Audio(data=combined, rate=SAMPLE_RATE, autoplay=True))